In [46]:
import pandas as pd
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score

In [3]:
df = pd.read_csv("../DataSets/NLP(ML).txt", sep = ";", header= 0, names=["Sentiment","emotion"])
df.head()

,Sentiment,emotion
0,i can go from feeling so hopeless to so damned...,sadness
1,im grabbing a minute to post i feel greedy wrong,anger
2,i am ever feeling nostalgic about the fireplac...,love
3,i am feeling grouchy,anger
4,ive been feeling a little burdened lately wasn...,sadness


In [9]:
df.isnull().sum()

Sentiment    0
emotion      0
dtype: int64

In [8]:
df["Sentiment"] = df["Sentiment"].apply(lambda x: x.lower())

In [10]:
unique_emotions = df["emotion"].unique()
emotion_number = {}
i = 0
for emo in unique_emotions:
    emotion_number[emo] = i
    i += 1

df["emotion"] = df["emotion"].map(emotion_number)

In [11]:
df.head()

,Sentiment,emotion
0,i can go from feeling so hopeless to so damned...,0
1,im grabbing a minute to post i feel greedy wrong,1
2,i am ever feeling nostalgic about the fireplac...,2
3,i am feeling grouchy,1
4,ive been feeling a little burdened lately wasn...,0


In [13]:
def remove_punc(txt):
    return txt.translate(str.maketrans('', '', string.punctuation))

In [14]:
df["Sentiment"] = df["Sentiment"].apply(lambda x: remove_punc(x))

In [15]:
def remove_num(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new
df["Sentiment"] = df["Sentiment"].apply(lambda x: remove_num(x))

In [17]:
def remove_emoji(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new
df["Sentiment"] = df["Sentiment"].apply(lambda x: remove_emoji(x))

In [19]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to C:\Users\hasnain
[nltk_data]     ali\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package stopwords to C:\Users\hasnain
[nltk_data]     ali\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [26]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to C:\Users\hasnain
[nltk_data]     ali\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [27]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(txt):
    words = word_tokenize(txt)
    cleaned = []
    for i in words:
        if not i in stop_words:
            cleaned.append(i)

    return ' '.join(cleaned)

In [28]:
df["Sentiment"] = df["Sentiment"].apply(lambda x: remove_stopwords(x))

In [29]:
df.loc[1]['Sentiment']

'im grabbing minute post feel greedy wrong'

In [30]:
df.head()

,Sentiment,emotion
0,go feeling hopeless damned hopeful around some...,0
1,im grabbing minute post feel greedy wrong,1
2,ever feeling nostalgic fireplace know still pr...,2
3,feeling grouchy,1
4,ive feeling little burdened lately wasnt sure,0


In [32]:
X_train, X_test, y_train, y_test = train_test_split(df["Sentiment"], df["emotion"], test_size = 0.2, random_state = 42)

 ## Vectorizor

### Bag of Words

In [33]:
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)

pred_bow = nb_model.predict(X_test_bow)
accuracy_score(y_test, pred_bow)

0.7653125

### Tfidf

In [40]:
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

nb_model_tf = MultinomialNB()
nb_model_tf.fit(X_train_tfidf, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [41]:
tfidf_pred = nb_model_tf.predict(X_test_tfidf)
accuracy_score(y_test,tfidf_pred)

0.7140625

#### I will be using grid search CV for better accuracy

In [42]:
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB

params = {
    "alpha": [0.001, 0.01, 0.1, 0.5, 1, 2, 5, 10],
    "fit_prior": [True, False]
}

grid = GridSearchCV(
    MultinomialNB(),
    params,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train_tfidf, y_train)

print("Best Params:", grid.best_params_)

Best Params: {'alpha': 0.5, 'fit_prior': False}


In [43]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test_tfidf)
accuracy_score(y_test,y_pred)

0.854375

In [44]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("nb", MultinomialNB())
])

params = {
    "tfidf__ngram_range": [(1,1), (1,2), (1,3)],
    "tfidf__min_df": [1, 2, 3, 5],
    "tfidf__max_df": [0.9, 0.95, 1.0],
    "tfidf__sublinear_tf": [True, False],
    "nb__alpha": [0.01, 0.05, 0.1, 0.5, 1, 2]
}

grid = GridSearchCV(
    pipeline,
    params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)
print("CV Accuracy:", grid.best_score_)

y_pred = grid.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred))

Best Params: {'nb__alpha': 0.1, 'tfidf__max_df': 0.9, 'tfidf__min_df': 5, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': True}
CV Accuracy: 0.8349877576690113
Test Accuracy: 0.835


In [45]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

model = MultinomialNB(alpha=0.1)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print(accuracy_score(y_test, y_pred))

0.8171875


## Logistic Regression

In [47]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )),
    ("lr", LogisticRegression(
        max_iter=1000
    ))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))


Accuracy: 0.865625


In [48]:
params = {
    "tfidf__ngram_range": [(1,1), (1,2), (1,3)],
    "tfidf__min_df": [1, 2, 3, 5],
    "tfidf__max_df": [0.9, 0.95, 1.0],
    "tfidf__sublinear_tf": [True, False],
    "lr__C": [0.01, 0.1, 0.5, 1, 2, 5, 10],
    "lr__solver": ["liblinear", "lbfgs"]
}

sm_params = {
    "tfidf__ngram_range": [(1,1), (1,2)],
    "tfidf__min_df": [1, 2, 5],
    "tfidf__max_df": [0.9, 0.95],
    "tfidf__sublinear_tf": [True, False],
    "lr__C": [0.1, 1, 10],
    "lr__solver": ["liblinear", "lbfgs"]
}

grid = GridSearchCV(
    pipeline,
    sm_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)
print("Best Params:", grid.best_params_)
print("CV Accuracy:", grid.best_score_)

y_pred = grid.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred))

C:\conda\anaconda3\envs\DSenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


Best Params: {'lr__C': 10, 'lr__solver': 'liblinear', 'tfidf__max_df': 0.9, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': True}
CV Accuracy: 0.898820156799531
Test Accuracy: 0.8978125


In [49]:
import joblib
joblib.dump(grid.best_estimator_, "emotion_model.pkl")

['emotion_model.pkl']